In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import gc
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast  # Mixed Precision for faster training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc, average_precision_score
from sklearn.cluster import KMeans
from scipy.stats import beta as beta_dist
import joblib
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# SMOTE for imbalanced data
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    print("⚠️ imbalanced-learn not installed. Run: !pip install -q imbalanced-learn")
    SMOTE_AVAILABLE = False

# =============================================================================
# TPU/GPU DETECTION (PyTorch)
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Platform: {platform.platform()}")

# TPU setup
TPU_AVAILABLE = False
xm = None
xmp = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xla_model
    TPU_AVAILABLE = True
    xm = xla_model
    print(f"✅ TPU Available: torch_xla {torch_xla.__version__}")
    try:
        import torch_xla.distributed.xla_multiprocessing as xla_mp
        xmp = xla_mp
    except ImportError:
        pass
except ImportError:
    print("⚠️ TPU not available (torch_xla not installed)")
    print("   For Kaggle TPU: Runtime → Change runtime type → TPU")

# CUDA setup
if torch.cuda.is_available():
    print(f"\n✅ CUDA Available")
    print(f"   GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB)")
else:
    print("\n⚠️ CUDA not available")

print("="*80 + "\n")

# Device selection
USE_XLA = False
DEVICE = None

if TPU_AVAILABLE and xm is not None:
    USE_XLA = True
    DEVICE = xm.xla_device()
    print(f"🎯 Using TPU: {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🎯 Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"🎯 Using CPU: {DEVICE}")

print("")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset - CICAPT-IIoT
DATA_FILES = [
    '/kaggle/input/cicapt-iiot/phase1_NetworkData.csv',
    '/kaggle/input/cicapt-iiot/phase2_NetworkData.csv'
]

# Columns to drop (metadata, identifiers)
DROP_COLS = [
    'ts', 'Source IP', 'Destination IP', 'Source Port', 'Destination Port', 
    'subLabel', 'subLabelCat', 'Protocol_name', 'MAC', 'Timestamp', 
    'Flow ID', 'Unnamed: 0', 'Fwd Header Length.1'
]

# Feature selection settings
USE_TOP_FEATURES = True  # Use Feature Selection with Random Forest
TOP_N_FEATURES = 20      # Number of selected features

# Sampling settings for large dataset (CICAPT has ~2M samples)
SAMPLE_NORMAL_FRAC = 0.2  # Take 20% normal traffic for better balance
BALANCE_SAMPLE_SIZE = 200000  # 200k samples for reliability

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 4096 if torch.cuda.is_available() else (1024 if TPU_AVAILABLE else 512)
LEARNING_RATE = 0.001
PATIENCE = 10


# =============================================================================
# MODEL SELECTION - MorphIDS Deep Learning Pool + Apollon
# =============================================================================
MODELS_TO_TRAIN = {
    'mlp_model': True,          # ✅ Deep MLP (MorphIDS)
    'cnn_model': True,          # ✅ 1D CNN (MorphIDS)
    'tcn_model': True,          # ✅ Temporal CNN (MorphIDS)
    'bilstm_model': True,       # ✅ BiLSTM-Attention (MorphIDS)
    'apollon_model': True,      # ✅ Apollon (MAB with MorphIDS Pool)
}


# =============================================================================
# MORPHIDS MODEL ARCHITECTURES (Deep Learning)
# =============================================================================

class MLPDropout(nn.Module):
    """Deep MLP with dropout for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, d_out)
        )
    def forward(self, x): return self.net(x)


class DeepCNN(nn.Module):
    """1D CNN for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.proj = nn.Linear(d_in, 64)
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, d_out))
    def forward(self, x):
        x = F.relu(self.proj(x)).unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(self.pool(x).squeeze(-1))


class DeepTCN(nn.Module):
    """Temporal Convolutional Network for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        self.tcn1 = nn.Conv1d(1, 16, kernel_size=3, padding=2, dilation=2)
        self.tcn2 = nn.Conv1d(16, 32, kernel_size=3, padding=4, dilation=4)
        self.tcn3 = nn.Conv1d(32, 32, kernel_size=3, padding=8, dilation=8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, d_out))
    def forward(self, x):
        x = F.relu(self.input(x)).unsqueeze(1)
        x = F.relu(self.tcn1(x))
        x = F.relu(self.tcn2(x))
        x = F.relu(self.tcn3(x))
        return self.fc(self.pool(x).squeeze(-1))


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention mechanism for IDS classification."""
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.proj_size = 64
        
        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(d_in, self.proj_size),
            nn.LayerNorm(self.proj_size),
            nn.ReLU(),
            nn.Dropout(p)
        )
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=self.proj_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=p if num_layers > 1 else 0
        )
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=num_heads,
            dropout=p,
            batch_first=True
        )
        
        # Output layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        
        # Project input
        x = self.input_proj(x)
        x = x.unsqueeze(1)  # (batch, 1, proj_size)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)  # (batch, 1, hidden*2)
        
        # Self-attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Global average pooling
        pooled = attn_out.mean(dim=1)  # (batch, hidden*2)
        
        # Classification
        logits = self.classifier(pooled)
        return logits


# =============================================================================
# PYTORCH MODEL WRAPPER (for Apollon compatibility)
# =============================================================================

class PyTorchModelWrapper:
    """Wrapper to make PyTorch models compatible with sklearn-like interface."""
    
    def __init__(self, model_class, d_in, d_out=2, device=None, epochs=50, batch_size=1024, lr=0.001):
        self.model_class = model_class
        self.d_in = d_in
        self.d_out = d_out
        self.device = device if device else DEVICE
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.model = None
        self.class_weights = None
        
    def fit(self, X, y):
        """Train the model."""
        # Convert to tensors
        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)
        
        # Calculate class weights
        class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
        self.class_weights = torch.FloatTensor(class_weights).to(self.device)
        
        # Create model
        self.model = self.model_class(self.d_in, self.d_out).to(self.device)
        
        # Create dataloader
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        # Training setup
        criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        
        # Train
        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        
        return self
    
    def predict(self, X):
        """Predict class labels with batched processing to avoid OOM."""
        self.model.eval()
        all_preds = []
        batch_size = self.batch_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = torch.FloatTensor(X[i:i+batch_size]).to(self.device)
                outputs = self.model(batch_X)
                _, predicted = torch.max(outputs, 1)
                all_preds.append(predicted.cpu())
                del batch_X, outputs  # Free memory
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        return torch.cat(all_preds).numpy()
    
    def predict_proba(self, X):
        """Predict class probabilities with batched processing to avoid OOM."""
        self.model.eval()
        all_probs = []
        batch_size = self.batch_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = torch.FloatTensor(X[i:i+batch_size]).to(self.device)
                outputs = self.model(batch_X)
                probs = F.softmax(outputs, dim=1)
                all_probs.append(probs.cpu())
                del batch_X, outputs  # Free memory
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        return torch.cat(all_probs).numpy()


# =============================================================================
# APOLLON ARCHITECTURE (Multi-Armed Bandit with MorphIDS Deep Learning Pool)
# =============================================================================

class MultiArmedBanditThompsonSampling:
    """
    Apollon with MorphIDS Deep Learning Pool
    
    This implementation uses Multi-Armed Bandits (MAB) with Thompson Sampling to dynamically
    select the optimal Deep Learning classifier for each input. Instead of traditional ML
    models, we use MorphIDS deep learning architectures (MLP, CNN, TCN, BiLSTM-Attention).
    
    Reference: Apollon paper - "Apollon: A Robust Defence System against Adversarial 
    Machine Learning Attacks in Intrusion Detection Systems"
    
    Modified to use MorphIDS pool for enhanced detection capabilities.
    """
    
    def __init__(self, d_in, n_arms=4, n_clusters=2, seed=42, device=None):
        """
        Initialize the Multi-Armed Bandit with Thompson Sampling using MorphIDS models.
        
        Parameters:
            d_in: Input dimension (number of features)
            n_arms: Number of classifier arms (default: 4 - MLP, CNN, TCN, BiLSTM)
            n_clusters: Number of clusters for input space partitioning
            seed: Random seed for reproducibility
            device: PyTorch device (CPU/CUDA/TPU)
        """
        self.d_in = d_in
        self.n_arms = n_arms
        self.n_clusters = n_clusters
        self.seed = seed
        self.device = device if device else DEVICE
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # Initialize MorphIDS Deep Learning arms
        self.arms = [
            PyTorchModelWrapper(MLPDropout, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepCNN, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepTCN, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(BiLSTMAttention, d_in, 2, self.device, epochs=30, batch_size=1024),
        ]
        self.arm_names = ['MLP', 'CNN', 'TCN', 'BiLSTM-Attention']
        
        self.cluster_centers = None
        self.cluster_assignments = None
        self.reward_sums = {}
        for cluster in range(n_clusters):
            self.reward_sums[cluster] = np.zeros(n_arms)
        
        # Beta distribution parameters for Thompson Sampling
        self.alpha = np.ones(self.n_arms)
        self.beta = np.ones(self.n_arms)
        
        self.training_time = 0.0
    
    def train(self, X_train, y_train):
        """
        Train all MorphIDS classifier arms and cluster the input space.
        
        Parameters:
            X_train: Training features
            y_train: Training labels
        """
        start_time = time.time()
        
        # Cluster the training data
        print(f"    Clustering input space into {self.n_clusters} clusters...")
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.cluster_assignments = kmeans.fit_predict(X_train)
        self.cluster_centers = kmeans.cluster_centers_
        
        # Print cluster distribution
        for i in range(self.n_clusters):
            cluster_count = np.sum(self.cluster_assignments == i)
            print(f"      Cluster {i}: {cluster_count:,} samples ({cluster_count/len(y_train)*100:.2f}%)")
        
        # Train all MorphIDS arms on the full training data
        print(f"\n    Training {self.n_arms} MorphIDS Deep Learning arms...")
        for arm_idx, (arm, name) in enumerate(zip(self.arms, self.arm_names)):
            print(f"      [{arm_idx+1}/{self.n_arms}] Training {name}...")
            arm.fit(X_train, y_train)
        
        # Calculate reward sums for each arm in each cluster
        print(f"\n    Computing arm rewards per cluster...")
        for cluster_idx in range(self.n_clusters):
            cluster_mask = self.cluster_assignments == cluster_idx
            cluster_X = X_train[cluster_mask]
            cluster_y = y_train[cluster_mask]
            
            for arm_idx, arm in enumerate(self.arms):
                if len(cluster_X) > 0:
                    arm_preds = arm.predict(cluster_X)
                    accuracy = np.mean(arm_preds == cluster_y)
                    self.reward_sums[cluster_idx][arm_idx] = accuracy
                    print(f"      Cluster {cluster_idx}, {self.arm_names[arm_idx]}: {accuracy*100:.2f}% accuracy")
        
        self.training_time = time.time() - start_time
        print(f"\n    ✅ Training completed in {self.training_time:.2f} seconds")
    
    def select_arm(self, cluster):
        """
        Select an arm using Thompson Sampling for the given cluster.
        
        Parameters:
            cluster: Cluster index
            
        Returns:
            Selected arm index
        """
        theta = np.zeros(self.n_arms)
        for arm_idx in range(self.n_arms):
            # Sample from Beta distribution
            alpha_param = self.alpha[arm_idx] + self.reward_sums[cluster][arm_idx]
            beta_param = self.beta[arm_idx] + 1 - self.reward_sums[cluster][arm_idx]
            theta[arm_idx] = np.random.beta(alpha_param, beta_param)
        return np.argmax(theta)
    
    def predict(self, X_test):
        """
        Predict using Thompson Sampling to dynamically select classifiers.
        
        Parameters:
            X_test: Test features
            
        Returns:
            predictions: Predicted labels
            selected_arms: Arms selected for each sample
        """
        # Assign each sample to a cluster
        clusters = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            distances = np.linalg.norm(self.cluster_centers - X_test[i], axis=1)
            clusters[i] = np.argmin(distances)
        
        # Select arm for each sample using Thompson Sampling
        selected_arms = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            selected_arms[i] = self.select_arm(clusters[i])
        
        # Predict using selected arms
        predictions = np.zeros(len(X_test), dtype=int)
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                predictions[arm_mask] = self.arms[arm_idx].predict(arm_X)
        
        return predictions, selected_arms
    
    def predict_proba(self, X_test):
        """
        Predict probabilities using Thompson Sampling.
        
        Parameters:
            X_test: Test features
            
        Returns:
            probabilities: Probability of positive class (Attack)
        """
        predictions, selected_arms = self.predict(X_test)
        
        # Get probabilities from each arm
        probabilities = np.zeros(len(X_test))
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                if hasattr(self.arms[arm_idx], 'predict_proba'):
                    probs = self.arms[arm_idx].predict_proba(arm_X)
                    if probs.shape[1] > 1:
                        probabilities[arm_mask] = probs[:, 1]
                    else:
                        probabilities[arm_mask] = probs[:, 0]
                else:
                    # For classifiers without predict_proba, use predictions
                    probabilities[arm_mask] = predictions[arm_mask]
        
        return probabilities


# =============================================================================
# DATA LOADING AND PREPROCESSING (CICAPT-IIoT)
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING AND PREPROCESSING - CICAPT-IIoT")
print("="*80)

# --- STEP 1: READ DATA WITH CHUNKING (Memory Efficient) ---
print("\n>>> STEP 1: LOADING DATA (CHUNKING)...")
processed_chunks = []
total_rows = 0

for file_path in DATA_FILES:
    if not os.path.exists(file_path):
        print(f"   ⚠️ File not found: {file_path}")
        continue
    print(f"   📂 Processing: {file_path}")
    
    # Read in chunks to save RAM
    with pd.read_csv(file_path, chunksize=1_000_000, low_memory=False) as reader:
        for chunk_idx, chunk in enumerate(reader):
            if chunk_idx == 0:
                print(f"      DEBUG: Unique labels in Chunk 1: {chunk['label'].unique()}")
                
            # Get all attack samples (ALL non-zero labels)
            attacks = chunk[chunk['label'] != 0]
            
            # Sample normal samples
            normals = chunk[chunk['label'] == 0].sample(frac=SAMPLE_NORMAL_FRAC, random_state=seed)
            
            processed_chunks.extend([attacks, normals])
            total_rows += len(chunk)
            print(f"      Chunk {chunk_idx+1}: {len(chunk):,} rows → Attack: {len(attacks):,}, Normal (sampled): {len(normals):,}")

# Concatenate chunks
df = pd.concat(processed_chunks, ignore_index=True)
del processed_chunks
gc.collect()
print(f"\n   ✅ Total rows read: {total_rows:,}")
print(f"   ✅ After sampling: {len(df):,} samples")
print(f"   📊 Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# --- STEP 2: DATA CLEANING ---
print("\n>>> STEP 2: DATA CLEANING...")

# Strip whitespace from column names
df.columns = df.columns.str.strip()
print(f"   Total columns: {len(df.columns)}")

# Drop metadata columns
cols_to_drop = [c for c in DROP_COLS if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"   Dropped {len(cols_to_drop)} metadata columns: {cols_to_drop}")

# Replace infinite values with NaN
inf_count = np.isinf(df.select_dtypes(include=[np.number]).values).sum()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
if inf_count > 0:
    print(f"   Infinite values replaced with NaN: {inf_count}")

# Drop NaN rows
nan_rows_before = len(df)
df.dropna(inplace=True)
nan_removed = nan_rows_before - len(df)
if nan_removed > 0:
    print(f"   Rows with NaN removed: {nan_removed}")

# Drop duplicates
dup_before = len(df)
df.drop_duplicates(inplace=True)
dup_removed = dup_before - len(df)
if dup_removed > 0:
    print(f"   Duplicate rows removed: {dup_removed}")

df.reset_index(drop=True, inplace=True)
print(f"   ✅ Final shape after cleaning: {df.shape}")
print(f"   📊 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# --- BALANCE FOR FEATURE SELECTION ---
print("\n>>> STEP 2.5: BALANCING FOR FEATURE SELECTION...")
df_atk = df[df['label'] == 1]
n_atk = len(df_atk)
n_norm_sample = min(BALANCE_SAMPLE_SIZE, len(df[df['label'] == 0]))
df_norm = df[df['label'] == 0].sample(n=n_norm_sample, random_state=seed)
df_balanced = pd.concat([df_norm, df_atk], ignore_index=True)
print(f"   Attack samples: {n_atk:,}")
print(f"   Normal samples (sampled): {n_norm_sample:,}")
print(f"   Balanced dataset: {len(df_balanced):,}")

# Inspect label distribution
print("\n" + "-"*80)
print("LABEL DISTRIBUTION")
print("-"*80)
print("\nAbsolute counts:")
print(df_balanced['label'].value_counts().to_string())
print(f"\nRelative proportions:")
for label, prop in df_balanced['label'].value_counts(normalize=True).items():
    label_name = "Attack" if label == 1 else "Normal"
    print(f"  {label_name:<20} {prop*100:>6.2f}%")
print(f"\nTotal unique labels: {df_balanced['label'].nunique()}")
print(f"Total samples: {len(df_balanced):,}")

# Extract features and target
print("\n" + "-"*80)
print("FEATURE EXTRACTION")
print("-"*80)
X = df_balanced.drop('label', axis=1).copy()
y = df_balanced['label'].copy()

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Features data type: {X.dtypes.value_counts().to_dict()}")
print(f"\nBinary class distribution:")
for class_label, count in pd.Series(y).value_counts().sort_index().items():
    class_name = "Normal" if class_label == 0 else "Attack"
    print(f"  Class {class_label} ({class_name}): {count:>10,} samples ({count/len(y)*100:>5.2f}%)")
print(f"\nClass imbalance ratio: {pd.Series(y).value_counts().max() / pd.Series(y).value_counts().min():.2f}:1")

# Free memory
del df, df_balanced, df_atk, df_norm
gc.collect()

# =============================================================================
# DATA SPLITTING
# =============================================================================
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    stratify=y,
    test_size=0.20,
    random_state=seed,
    shuffle=True
)

# Split val from train: 70% train, 10% val from total
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    stratify=y_train,
    test_size=0.125,  # 0.125 of 80% = 10% of total
    random_state=seed,
    shuffle=True
)

print(f"\nDataset split results:")
print(f"  Train set: {X_train.shape[0]:>10,} samples ({X_train.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Val set:   {X_val.shape[0]:>10,} samples ({X_val.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Test set:  {X_test.shape[0]:>10,} samples ({X_test.shape[0]/len(X)*100:>5.2f}%)")
print(f"\nClass distribution in splits:")
print(f"  Train - Normal: {(y_train==0).sum():>10,}, Attack: {(y_train==1).sum():>10,}")
print(f"  Val   - Normal: {(y_val==0).sum():>10,}, Attack: {(y_val==1).sum():>10,}")
print(f"  Test  - Normal: {(y_test==0).sum():>10,}, Attack: {(y_test==1).sum():>10,}")

# =============================================================================
# SMOTE RESAMPLING (Optional - on training data only)
# =============================================================================
if SMOTE_AVAILABLE:
    print("\n" + "="*80)
    print("SMOTE RESAMPLING (TRAINING DATA ONLY)")
    print("="*80)
    
    print(f"\nBefore SMOTE:")
    print(f"  Train - Normal: {(y_train==0).sum():>10,}, Attack: {(y_train==1).sum():>10,}")
    
    smote = SMOTE(sampling_strategy=0.7, random_state=seed)  # 7:10 ratio (minority:majority)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    print(f"\nAfter SMOTE (sampling_strategy=0.7):")
    print(f"  Train - Normal: {(y_train_resampled==0).sum():>10,}, Attack: {(y_train_resampled==1).sum():>10,}")
    print(f"  Total samples: {len(X_train_resampled):,}")
    
    # Use SMOTE data for training
    X_train = X_train_resampled
    y_train = y_train_resampled
else:
    print("\n⚠️ SMOTE not available - training with original imbalanced data")

# =============================================================================
# FEATURE SELECTION (Top 20 Features using Random Forest)
# =============================================================================
from sklearn.ensemble import RandomForestClassifier as skRF

selected_features = None
original_feature_names = list(X.columns)

if USE_TOP_FEATURES:
    print("\n" + "="*80)
    print("FEATURE SELECTION (Random Forest Importance)")
    print("="*80)
    
    print(f"\nTraining temporary RF model for feature selection...")
    temp_rf = skRF(n_estimators=50, random_state=seed, n_jobs=-1)
    temp_rf.fit(X_train, y_train)
    
    # Get feature importances
    importances = temp_rf.feature_importances_
    indices = np.argsort(importances)[::-1]
    selected_features = [original_feature_names[i] for i in indices[:TOP_N_FEATURES]]
    
    print(f"\n✅ Top {TOP_N_FEATURES} Features Selected:")
    for i, feat in enumerate(selected_features, 1):
        print(f"   {i:2d}. {feat} (importance: {importances[indices[i-1]]:.4f})")
    
    # Filter datasets - convert to DataFrame if needed
    if isinstance(X_train, pd.DataFrame):
        X_train = X_train[selected_features]
    else:
        # X_train is numpy array (after SMOTE)
        X_train_df = pd.DataFrame(X_train, columns=original_feature_names)
        X_train = X_train_df[selected_features]
    
    X_val = X_val[selected_features]
    X_test = X_test[selected_features]
    
    print(f"\n   Filtered datasets:")
    print(f"   Train: {X_train.shape}")
    print(f"   Val:   {X_val.shape}")
    print(f"   Test:  {X_test.shape}")
    
    # Save features list
    features_path = os.path.join(OUT_DIR, 'selected_features.txt')
    with open(features_path, 'w') as f:
        for feature in selected_features:
            f.write(f"{feature}\n")
    print(f"   💾 Selected features saved: {features_path}")
    
    del temp_rf
    gc.collect()

# =============================================================================
# DATA SCALING
# =============================================================================
print("\n" + "="*80)
print("DATA SCALING (Z-SCORE NORMALIZATION)")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train set scaled: {X_train_scaled.shape}")
print(f"Val set scaled:   {X_val_scaled.shape}")
print(f"Test set scaled:  {X_test_scaled.shape}")

# Get input dimension
input_size = X_train_scaled.shape[1]
print(f"\n📊 Input dimension: {input_size} features")

# Save scaler for inference
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

# Convert labels to numpy arrays
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_val_np = y_val.values if hasattr(y_val, 'values') else y_val
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test


# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================

def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(8, 6))
    
    # Normalize confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create annotations with both counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', 
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'],
                cbar_kws={'label': 'Proportion'},
                linewidths=1, linecolor='gray')
    
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    📊 Confusion matrix plot saved: {save_path}")
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve and save with AUC annotation."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc*100:.2f}%)')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 ROC curve plot saved: {save_path} (AUC={roc_auc*100:.2f}%)")
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve and save with Average Precision (AP) annotation."""
    precision_vals, recall_vals, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='purple', lw=2, label=f'PR curve (AP = {ap*100:.2f}%)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'{model_name} - Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 PR curve plot saved: {save_path} (AP={ap*100:.2f}%)")
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot comparison of all models across different metrics."""
    metrics = ['accuracy', 'recall', 'f1', 'auc']
    metric_names = ['Accuracy', 'Detection Rate', 'F1 Score', 'AUC-ROC']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    # Extended color palette for 6 models
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    for i, model_name in enumerate(model_names):
        color = colors[i % len(colors)]
        values = [results[model_name][metric] * 100 for metric in metrics]
        offset = (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model_name, color=color, alpha=0.8)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%',
                   ha='center', va='bottom', fontsize=9)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 105])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n📊 Model comparison plot saved: {save_path}")
    plt.close()


# =============================================================================
# MODEL TRAINING
# =============================================================================
print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

trained_models = {}
results = {}

# Convert to PyTorch tensors
print("\n⚙️ Converting to PyTorch tensors...")
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train_np)
X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val_np)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test_np)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")


# Train Apollon (MAB with MorphIDS Deep Learning Pool)
if MODELS_TO_TRAIN.get('apollon_model', False):
    print(f"\n{'='*80}")
    print(f"TRAINING: Apollon (MAB with MorphIDS Deep Learning Pool)")
    print(f"{'='*80}")
    print(f"Training samples: {len(y_train_np):,}")
    print(f"Test samples: {len(y_test_np):,}")
    print(f"Number of clusters: 2")
    print(f"Number of arms (DL models): 4 (MLP, CNN, TCN, BiLSTM-Attention)")
    print("-" * 80)
    
    # Initialize and train Apollon with MorphIDS pool
    apollon = MultiArmedBanditThompsonSampling(d_in=input_size, n_arms=4, n_clusters=2, seed=seed, device=DEVICE)
    apollon.train(X_train_scaled, y_train_np)
    
    # Predict
    start_pred = time.time()
    apollon_preds, selected_arms = apollon.predict(X_test_scaled)
    pred_time = time.time() - start_pred
    print(f"\n    Prediction time: {pred_time:.4f} seconds")
    
    # Get probabilities
    apollon_probs = apollon.predict_proba(X_test_scaled)
    
    # Calculate metrics
    apollon_acc = accuracy_score(y_test_np, apollon_preds)
    apollon_prec = precision_score(y_test_np, apollon_preds, zero_division=0)
    apollon_rec = recall_score(y_test_np, apollon_preds, zero_division=0)
    apollon_f1 = f1_score(y_test_np, apollon_preds, zero_division=0)
    apollon_auc = roc_auc_score(y_test_np, apollon_probs)
    
    # Arm selection statistics
    print(f"\n    Arm Selection Statistics:")
    for arm_idx, arm_name in enumerate(apollon.arm_names):
        count = np.sum(selected_arms == arm_idx)
        print(f"      {arm_name}: {count:,} ({count/len(y_test_np)*100:.2f}%)")
    
    print(f"\n📊 Apollon Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {apollon_acc*100:>6.2f}%")
    print(f"    Detection Rate: {apollon_rec*100:>6.2f}%")
    print(f"    F1 Score:       {apollon_f1*100:>6.2f}%")
    print(f"    AUC-ROC:        {apollon_auc*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    apollon_cm = confusion_matrix(y_test_np, apollon_preds)
    print(f"                Predicted")
    print(f"                Normal  Attack")
    print(f"    Actual Normal  {apollon_cm[0,0]:>6,}  {apollon_cm[0,1]:>6,}")
    print(f"           Attack  {apollon_cm[1,0]:>6,}  {apollon_cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(y_test_np, apollon_preds, target_names=['Normal', 'Attack'], digits=4))
    
    # Store results
    results['Apollon'] = {
        'accuracy': apollon_acc,
        'precision': apollon_prec,
        'recall': apollon_rec,
        'f1': apollon_f1,
        'auc': apollon_auc,
        'predictions': apollon_preds,
        'probabilities': apollon_probs,
        'labels': y_test_np,
        'confusion_matrix': apollon_cm,
        'selected_arms': selected_arms
    }
    
    # Save Apollon model
    joblib.dump(apollon, os.path.join(OUT_DIR, 'apollon_model.pkl'))
    print(f"\n💾 Apollon model saved to: {os.path.join(OUT_DIR, 'apollon_model.pkl')}")
    
    # Generate plots
    plot_confusion_matrix(apollon_cm, 'Apollon (MorphIDS Pool)', os.path.join(PLOT_DIR, 'apollon_confusion_matrix.png'))
    plot_roc_curve(y_test_np, apollon_probs, 'Apollon (MorphIDS Pool)', os.path.join(PLOT_DIR, 'apollon_roc_curve.png'))
    plot_pr_curve(y_test_np, apollon_probs, 'Apollon (MorphIDS Pool)', os.path.join(PLOT_DIR, 'apollon_pr_curve.png'))


# =============================================================================
# FINAL RESULTS SUMMARY
# =============================================================================
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

if results:
    # Find best model by F1
    best_model_name = max(results, key=lambda k: results[k]['f1'])
    best_f1 = results[best_model_name]['f1']
    print(f"\n🏆 Best Model by F1 Score: {best_model_name} (F1: {best_f1*100:.2f}%)")
    
    print(f"\n✅ All models saved to: {OUT_DIR}")
    print(f"✅ All plots saved to: {PLOT_DIR}")
else:
    print("\n⚠️ No models were trained. Check MODELS_TO_TRAIN configuration.")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80 + "\n")



HARDWARE DETECTION
PyTorch: 2.8.0+cu126
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
⚠️ TPU not available (torch_xla not installed)
   For Kaggle TPU: Runtime → Change runtime type → TPU

✅ CUDA Available
   GPU Count: 2
   GPU 0: Tesla T4 (15.64 GB)
   GPU 1: Tesla T4 (15.64 GB)

🎯 Using CUDA: cuda


DATA LOADING AND PREPROCESSING - CICAPT-IIoT

>>> STEP 1: LOADING DATA (CHUNKING)...
   📂 Processing: /kaggle/input/cicapt-iiot/phase1_NetworkData.csv
      DEBUG: Unique labels in Chunk 1: [0]
      Chunk 1: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 2: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 3: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 4: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 5: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 6: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 7: 1,000,000 rows → Attack: 0, Normal (sampled): 200,000
      Chunk 8: 1,